# FIT-VTON: End-to-End Demo

This notebook demonstrates the full FIT-VTON pipeline:
1. Environment setup
2. Download base model (IDM-VTON)
3. Generate synthetic mini-dataset
4. Run inference with different fit deltas
5. Visualise results grid

**Requirements:** GPU with ≥8 GB VRAM recommended.

In [ ]:
# ── Cell 1: Setup ──────────────────────────────────────────────────────────
import subprocess, sys, os
from pathlib import Path

# Make sure we are in the repo root
repo_root = Path(".").resolve().parent
if (repo_root / "setup.py").exists():
    os.chdir(str(repo_root))
print(f"Working directory: {Path('.').resolve()}")

# Install the package if not already installed
try:
    import fit_vton
    print(f"fit_vton v{fit_vton.__version__} already installed")
except ImportError:
    print("Installing fit_vton ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", ".", "-q"])
    import fit_vton
    print(f"Installed fit_vton v{fit_vton.__version__}")

In [ ]:
# ── Cell 2: Imports ────────────────────────────────────────────────────────
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image, ImageDraw

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nUsing device: {DEVICE}")

In [ ]:
# ── Cell 3: Download Base Model ────────────────────────────────────────────
# This downloads IDM-VTON (~10 GB) from HuggingFace.
# Skip if you already have it cached.

import subprocess

MODEL_DIR = "cache/models"
MODEL_ID = "yisol/IDM-VTON"

# Check if already downloaded
from pathlib import Path
local_dir = Path(MODEL_DIR) / MODEL_ID.replace("/", "--")
if local_dir.exists() and any(local_dir.iterdir()):
    print(f"Model already available at: {local_dir}")
else:
    print("Downloading IDM-VTON ... (this may take a few minutes)")
    result = subprocess.run(
        [sys.executable, "scripts/download_base_model.py",
         "--model_id", MODEL_ID,
         "--output_dir", MODEL_DIR],
        capture_output=False
    )
    if result.returncode != 0:
        print("Download failed. Will use HuggingFace auto-download during inference.")

In [ ]:
# ── Cell 4: Generate Synthetic Mini-Dataset ────────────────────────────────
# Generates placeholder person/garment/tryon images with realistic measurements.
# 200 train / 50 val / 50 test samples (quick demo size).

DATASET_DIR = "data/fit_dataset_demo"

if Path(DATASET_DIR).exists() and (Path(DATASET_DIR) / "metadata_train.json").exists():
    print(f"Demo dataset already exists at: {DATASET_DIR}")
else:
    print("Generating synthetic demo dataset ...")
    subprocess.run(
        [sys.executable, "scripts/download_fit_dataset.py",
         "--mini",
         "--output_dir", DATASET_DIR,
         "--train_size", "200",
         "--val_size", "50",
         "--test_size", "50"],
        check=True
    )
    print("Done!")

# Peek at metadata
import json
with open(f"{DATASET_DIR}/metadata_train.json") as f:
    meta = json.load(f)

print(f"\nDataset: {len(meta)} training samples")
print("Sample metadata entry:")
import pprint
pprint.pprint(meta[0])

In [ ]:
# ── Cell 5: Inspect Dataset Samples ───────────────────────────────────────
from fit_vton.data.fit_dataset import FITDataset, FITDatasetConfig
from fit_vton.data.transforms import denormalize

cfg = FITDatasetConfig(
    data_dir=DATASET_DIR,
    split="train",
    image_size=256,
    use_augmentation=False,
    max_samples=8,
)
dataset = FITDataset(cfg)
print(f"Dataset size: {len(dataset)}")

# Show first 4 triplets
fig, axes = plt.subplots(4, 3, figsize=(12, 14))
fig.suptitle("Dataset Samples: Person | Garment | Try-on", fontsize=14, y=1.01)

SIZE_LABELS = ["XS", "S", "M", "L", "XL", "XXL"]

for i in range(4):
    sample = dataset[i]
    
    def t2img(t):
        return (denormalize(t).permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    
    delta = float(sample["fit_delta"].item())
    g_size = SIZE_LABELS[int(sample["garment_size_idx"].item())]
    
    axes[i, 0].imshow(t2img(sample["person_image"]))
    axes[i, 0].set_title(f"Person (sample {i})")
    axes[i, 0].axis("off")
    
    axes[i, 1].imshow(t2img(sample["garment_image"]))
    axes[i, 1].set_title(f"Garment (size={g_size})")
    axes[i, 1].axis("off")
    
    axes[i, 2].imshow(t2img(sample["tryon_image"]))
    axes[i, 2].set_title(f"Try-on (Δ={delta:+.0f})")
    axes[i, 2].axis("off")

plt.tight_layout()
plt.savefig("outputs/demo_dataset_samples.png", bbox_inches="tight", dpi=100)
plt.show()
print("Saved: outputs/demo_dataset_samples.png")

In [ ]:
# ── Cell 6: Inspect MeasurementEncoder ────────────────────────────────────
from fit_vton.models.measurement_encoder import MeasurementEncoder

encoder = MeasurementEncoder(
    embed_dim=768,
    hidden_dim=512,
    num_tokens=4,
    num_garment_sizes=6,
)

print(encoder)
print(f"\nParameters: {encoder.num_parameters:,}")

# Test forward pass
B = 4
body_meas = torch.randn(B, 5)                          # normalised measurements
garment_size = torch.randint(0, 6, (B,))               # size index
fit_delta = torch.tensor([-2., -1., 0., 1.])           # -2 to +1

tokens = encoder(body_meas, garment_size, fit_delta)
print(f"\nInput shapes:")
print(f"  body_measurements : {body_meas.shape}")
print(f"  garment_size_idx  : {garment_size.shape}")
print(f"  fit_delta         : {fit_delta.shape}")
print(f"\nOutput shape: {tokens.shape}  (B={B}, num_tokens=4, embed_dim=768)")

# Check that different fit deltas produce different tokens
tight_tokens = tokens[0]   # delta = -2 (too tight)
baggy_tokens = tokens[3]   # delta = +1 (too baggy)
cosine_sim = torch.nn.functional.cosine_similarity(
    tight_tokens.flatten().unsqueeze(0),
    baggy_tokens.flatten().unsqueeze(0),
).item()
print(f"\nCosine similarity (tight vs baggy tokens): {cosine_sim:.4f}")
print("(Should be <1 — different fit conditions produce different tokens)")

In [ ]:
# ── Cell 7: Run Inference (without loading full IDM-VTON) ──────────────────
# We demonstrate the measurement encoding and show synthetic try-on results.
# Full inference requires the IDM-VTON model download (~10 GB).

# For demo purposes, we simulate inference by using the synthetic dataset images
# and the MeasurementEncoder to show that different fit conditions produce
# different conditioning vectors.

from fit_vton.data.fit_dataset import MEASUREMENT_MEAN, MEASUREMENT_STD, SIZE_LABELS

# Example: 170cm, 65kg woman wearing an L garment (body size M)
body_measurements_raw = torch.tensor([[170.0, 65.0, 90.0, 72.0, 97.0]])
body_measurements_norm = (body_measurements_raw - MEASUREMENT_MEAN) / MEASUREMENT_STD

scenarios = [
    {"garment_size": "XS", "body_size": "M", "delta": -3},
    {"garment_size": "S",  "body_size": "M", "delta": -2},
    {"garment_size": "M",  "body_size": "M", "delta":  0},
    {"garment_size": "L",  "body_size": "M", "delta": +1},
    {"garment_size": "XL", "body_size": "M", "delta": +2},
]

print("Measurement token norms for different fit scenarios:")
print(f"{'Scenario':<30} {'Token norm':>12} {'Token[0,:4]'}")
print("-" * 70)

for s in scenarios:
    g_idx = SIZE_LABELS.index(s["garment_size"])
    g_tensor = torch.tensor([g_idx])
    delta_tensor = torch.tensor([[float(s["delta"])]])
    
    with torch.no_grad():
        tokens = encoder(body_measurements_norm, g_tensor, delta_tensor)
    
    norm = tokens.norm().item()
    first4 = tokens[0, 0, :4].tolist()
    label = f"Body {s['body_size']} + Garment {s['garment_size']} (Δ={s['delta']:+d})"
    print(f"{label:<30} {norm:>12.4f} {[f'{v:.3f}' for v in first4]}")

In [ ]:
# ── Cell 8: Full Inference Demo (requires downloaded IDM-VTON) ─────────────
# Uncomment and run if you have downloaded the base model.

RUN_FULL_INFERENCE = False  # set to True after downloading IDM-VTON

if RUN_FULL_INFERENCE:
    from fit_vton.models.pipeline import FitVTONPipeline
    
    pipeline = FitVTONPipeline.from_pretrained(
        base_model_id="yisol/IDM-VTON",
        device=DEVICE,
    )
    print(f"Trainable parameters: {pipeline.num_trainable_parameters:,}")
    
    # Load a sample from the dataset
    sample = dataset[0]
    from fit_vton.data.transforms import denormalize
    
    def t2pil(t):
        arr = (denormalize(t).permute(1, 2, 0).numpy() * 255).astype(np.uint8)
        return Image.fromarray(arr)
    
    person_pil = t2pil(sample["person_image"])
    garment_pil = t2pil(sample["garment_image"])
    
    # Decode measurements
    body_meas_np = sample["body_measurements"].numpy()
    raw = body_meas_np * MEASUREMENT_STD.numpy() + MEASUREMENT_MEAN.numpy()
    height, weight, chest, waist, hip = raw
    
    garment_size_idx = int(sample["garment_size_idx"].item())
    fit_delta = float(sample["fit_delta"].item())
    body_size_idx = int(np.clip(garment_size_idx - round(fit_delta), 0, 5))
    
    results = {}
    for scale, label in [(0.0, "IDM-VTON (baseline)"), (1.0, "FIT-VTON (ours)")]:
        img = pipeline(
            person_image=person_pil,
            garment_image=garment_pil,
            body_height=float(height),
            body_weight=float(weight),
            body_chest=float(chest),
            body_waist=float(waist),
            body_hip=float(hip),
            body_size=SIZE_LABELS[body_size_idx],
            garment_size=SIZE_LABELS[garment_size_idx],
            num_inference_steps=20,
            seed=42,
            adapter_scale=scale,
        )
        results[label] = img
    
    # Display
    fig, axes = plt.subplots(1, 4, figsize=(16, 5))
    titles = ["Person", "Garment", "IDM-VTON (baseline)", "FIT-VTON (ours)"]
    imgs = [person_pil, garment_pil, results["IDM-VTON (baseline)"], results["FIT-VTON (ours)"]]
    
    for ax, img, title in zip(axes, imgs, titles):
        ax.imshow(img)
        ax.set_title(title, fontsize=12)
        ax.axis("off")
    
    fit_str = f'Fit Δ={fit_delta:+.0f} ({SIZE_LABELS[body_size_idx]} body → {SIZE_LABELS[garment_size_idx]} garment)'
    fig.suptitle(fit_str, fontsize=13)
    plt.tight_layout()
    plt.savefig("outputs/demo_inference.png", dpi=100)
    plt.show()
else:
    print("Full inference skipped (RUN_FULL_INFERENCE = False).")
    print("Set RUN_FULL_INFERENCE = True and run this cell after downloading IDM-VTON.")

In [ ]:
# ── Cell 9: Visualise Fit Delta Distribution in Dataset ────────────────────
import matplotlib.pyplot as plt
import json
import numpy as np

with open(f"{DATASET_DIR}/metadata_train.json") as f:
    meta = json.load(f)

fit_deltas = [m["fit_delta"] for m in meta]
body_sizes = [m["body_size_idx"] for m in meta]
garment_sizes = [m["garment_size_idx"] for m in meta]
heights = [m["height_cm"] for m in meta]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Dataset Statistics", fontsize=14)

# Fit delta distribution
axes[0].hist(fit_deltas, bins=range(-4, 5), color="steelblue", edgecolor="white", rwidth=0.8)
axes[0].set_xlabel("Fit Delta (garment - body size)")
axes[0].set_ylabel("Count")
axes[0].set_title("Fit Delta Distribution")
axes[0].axvline(0, color="red", linestyle="--", label="Perfect fit")
axes[0].legend()

# Body size distribution
SIZE_LABELS_PLOT = ["XS", "S", "M", "L", "XL", "XXL"]
from collections import Counter
body_counts = Counter(body_sizes)
axes[1].bar([SIZE_LABELS_PLOT[i] for i in sorted(body_counts)],
            [body_counts[i] for i in sorted(body_counts)],
            color="coral", edgecolor="white")
axes[1].set_xlabel("Body Size")
axes[1].set_ylabel("Count")
axes[1].set_title("Body Size Distribution")

# Height distribution
axes[2].hist(heights, bins=20, color="mediumseagreen", edgecolor="white")
axes[2].set_xlabel("Height (cm)")
axes[2].set_ylabel("Count")
axes[2].set_title("Height Distribution")
axes[2].axvline(np.mean(heights), color="red", linestyle="--",
               label=f"Mean={np.mean(heights):.0f}cm")
axes[2].legend()

plt.tight_layout()
Path("outputs").mkdir(exist_ok=True)
plt.savefig("outputs/demo_dataset_stats.png", dpi=100)
plt.show()
print("Saved: outputs/demo_dataset_stats.png")

In [ ]:
# ── Cell 10: Summary ───────────────────────────────────────────────────────
print("=" * 60)
print("FIT-VTON Demo Complete!")
print("=" * 60)
print()
print("What we demonstrated:")
print("  1. Synthetic FIT dataset generation")
print("  2. MeasurementEncoder: maps body/garment measurements to tokens")
print("  3. Different fit conditions → distinct conditioning vectors")
print("  4. Dataset statistics visualisation")
print()
print("Next steps:")
print("  - Download IDM-VTON: python scripts/download_base_model.py")
print("  - Start training:    python train.py --config configs/train.yaml")
print("  - Run inference:     python inference.py --help")
print("  - Evaluate:          python evaluate.py --help")
print()
print("GitHub: https://github.com/NatalieCarlebach1/fit-vton")